# 1. Base Setup

## 1.1 Install packages

In [21]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
!pip install kagglehub
!pip install statsmodels
!pip install xgboost
!pip install catboost

## 1.2 Load all needed imports

In [23]:
from pathlib import Path

import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
import numpy as np
import matplotlib.pyplot as plt

from dateutil.relativedelta import relativedelta

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, log_loss, confusion_matrix
from sklearn.calibration import calibration_curve
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit, GridSearchCV

from scipy.stats import randint
import sys, os
sys.path.append(os.path.abspath(".."))

# 2. Data Cleaning and preprocessing


In [24]:
def load_cashflow_data(csv_name: str = "dataset.csv") -> pd.DataFrame:
    """Load invoice dataset from local raw_data folder, or download from Kaggle.

    Args:
        csv_name: filename of the CSV to load.

    Returns:
        A pandas DataFrame with the raw invoice data.
    """
    base_dir = Path.cwd().parent
    raw_data_dir = base_dir / "raw_data"
    raw_data_dir.mkdir(parents=True, exist_ok=True)
    local_path = raw_data_dir / csv_name

    if local_path.is_file():
        print(f"Loading local file: {local_path}")
        return pd.read_csv(local_path)

    print("Local file not found, downloading from Kaggle via kagglehub...")
    df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        "pradumn203/payment-date-prediction-for-invoices-dataset",
        "dataset.csv",
    )

    df.to_csv(local_path, index=False)
    print(f"Saved local copy to {local_path}")

    return df

In [25]:
from cf_copilot.ml_logic.data import data_cleaning, build_sliding_window_snapshots
from cf_copilot.ml_logic.encoders import preprocess, NUMERIC_FEATURES, CATEGORICAL_FEATURES
from cf_copilot.ml_logic.model import initialize_model, train_model

hist_df = load_cashflow_data()

df = data_cleaning(hist_df)
big_df = build_sliding_window_snapshots(df)

big_df = big_df.sort_values("invoice_sent").reset_index(drop=True)
cutoff_date = big_df["invoice_sent"].quantile(0.8)

train_df = big_df[big_df["reference_date"] <= cutoff_date]
test_df = big_df[big_df["reference_date"] > cutoff_date]

X_train, y_train = preprocess(train_df)
X_test, y_test = preprocess(test_df)

Loading local file: /Users/anton/code/DERNTOAN/cf_copilot/raw_data/dataset.csv
Original rows: 39152
Augmented rows: 98169
week_bucket
1    38825
2    33060
3    10401
4     4009
5     3172
6     2192
7     6510
Name: count, dtype: int64


In [26]:
numeric_transformer = SimpleImputer(strategy="median")

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1)),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUMERIC_FEATURES),
        ("cat", categorical_transformer, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)

preprocessor

ColumnTransformer(transformers=[('num', SimpleImputer(strategy='median'),
                                 ['business_year', 'invoice_age_days',
                                  'days_until_due', 'pay_terms_days',
                                  'invoice_month', 'due_month', 'days_past_due',
                                  'customer_avg_delay', 'late_payment_ratio',
                                  'prev_transaction_count',
                                  'days_since_last_invoice',
                                  'customer_risk_score', 'invoice_amount',
                                  'invoice_amount_log']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value=-1,
                                                                strategy='constant')),
                                                 ('encoder',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1))]),
                                 ['invoice_currency', 'document_type',
                                  'invoice_size_cat', 'invoice_size_cat_q'])])

In [32]:
from xgboost import XGBClassifier
from sklearn.feature_selection import VarianceThreshold

classifier = XGBClassifier()

XGB_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("variance", VarianceThreshold(threshold=0.05)),
    ("classifier", classifier),
])

In [28]:
cutoff_date = big_df["reference_date"].quantile(0.8)
test_fold = np.where(big_df["reference_date"] <= cutoff_date, -1, 0)
ps = PredefinedSplit(test_fold)

X, y = preprocess(big_df)

In [30]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import LabelEncoder

# This is XGBoost specific
le = LabelEncoder()
y_enc = le.fit_transform(y)

param_grids = {
    "RandomForest": {
        "classifier__n_estimators": [100, 300, 500, 800],
        "classifier__max_depth": [5, 8, 10, 15, 20, None],
        "classifier__min_samples_split": [2, 5, 10, 20],
        "classifier__min_samples_leaf": [1, 5, 10, 20],
        "classifier__max_features": ["sqrt", "log2", 0.3, 0.5],
        "classifier__class_weight": ["balanced", "balanced_subsample", None],
    },
    "XGBoost": {
        "classifier__n_estimators": [100, 300, 500, 800],
        "classifier__max_depth": [3, 4, 6, 8, 10],
        "classifier__learning_rate": [0.01, 0.05, 0.1, 0.2],
        "classifier__subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
        "classifier__colsample_bytree": [0.5, 0.6, 0.8, 1.0],
        "classifier__reg_alpha": [0, 0.01, 0.1, 1.0, 10.0],
        "classifier__reg_lambda": [0.1, 1.0, 5.0, 10.0],
        "classifier__min_child_weight": [1, 3, 5, 10],
        "classifier__gamma": [0, 0.1, 0.5, 1.0],
        "classifier__scale_pos_weight": [1, 3, 5, 10],
    },
    "GradientBoosting": {
        "classifier__n_estimators": [100, 300, 500, 800],
        "classifier__max_depth": [3, 4, 5, 6, 8],
        "classifier__learning_rate": [0.01, 0.05, 0.1, 0.2],
        "classifier__subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
        "classifier__min_samples_split": [2, 5, 10, 20],
        "classifier__min_samples_leaf": [1, 5, 10, 20],
        "classifier__max_features": ["sqrt", "log2", 0.3, 0.5],
    },
}

model_name = "XGBoost"

tscv = TimeSeriesSplit(n_splits=5)

xgb_random_search = RandomizedSearchCV(
    XGB_pipeline,
    param_distributions=param_grids[model_name],
    n_iter=50,
    cv=tscv,
    scoring="neg_log_loss",
    n_jobs=-1,
    verbose=1,
    random_state=42,
)

xgb_random_search.fit(X, y_enc)
print(f"Best log_loss: {-xgb_random_search.best_score_:.4f}")
print(f"Best params: {xgb_random_search.best_params_}")


Fitting 5 folds for each of 50 candidates, totalling 250 fits


/Users/anton/.pyenv/versions/3.10.6/envs/cf_copilot/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [18:01:18] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/anton/.pyenv/versions/3.10.6/envs/cf_copilot/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [18:01:18] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/anton/.pyenv/versions/3.10.6/envs/cf_copilot/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [18:01:18] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/anton/.pyenv/versions/3.10.6/envs/cf_copilot/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [18:01:18] WAR

Best log_loss: 0.7127
Best params: {'classifier__subsample': 0.9, 'classifier__scale_pos_weight': 1, 'classifier__reg_lambda': 10.0, 'classifier__reg_alpha': 0.01, 'classifier__n_estimators': 800, 'classifier__min_child_weight': 5, 'classifier__max_depth': 8, 'classifier__learning_rate': 0.01, 'classifier__gamma': 0.5, 'classifier__colsample_bytree': 0.6}


In [ ]:
from sklearn.ensemble import RandomForestClassifier

classifier = RandomForestClassifier()

random_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("variance", VarianceThreshold(threshold=0.05)),
    ("classifier", classifier),
])
model_name = "RandomForest"

tscv = TimeSeriesSplit(n_splits=5)

random_random_search = RandomizedSearchCV(
    random_pipeline,
    param_distributions=param_grids[model_name],
    n_iter=50,
    cv=tscv,
    scoring="neg_log_loss",
    n_jobs=-1,
    verbose=1,
    random_state=42,
)

random_random_search.fit(X, y_enc)
print(f"Best log_loss: {-random_random_search.best_score_:.4f}")
print(f"Best params: {random_random_search.best_params_}")


Fitting 5 folds for each of 50 candidates, totalling 250 fits


/Users/anton/.pyenv/versions/3.10.6/envs/cf_copilot/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

classifier = GradientBoostingClassifier()

gradient_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("variance", VarianceThreshold(threshold=0.05)),
    ("classifier", classifier),
])
model_name = "GradientBoosting"

tscv = TimeSeriesSplit(n_splits=5)

gradient_random_search = RandomizedSearchCV(
    gradient_pipeline,
    param_distributions=param_grids[model_name],
    n_iter=50,
    cv=tscv,
    scoring="neg_log_loss",
    n_jobs=-1,
    verbose=1,
    random_state=42,
)

gradient_random_search.fit(X, y_enc)
print(f"Best log_loss: {-gradient_random_search.best_score_:.4f}")
print(f"Best params: {gradient_random_search.best_params_}")
